In [19]:
from llama_parse import LlamaParse
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import MarkdownElementNodeParser
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.retrievers import RecursiveRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

import os
from dotenv import load_dotenv 

In [ ]:
from docling.datamodel.pipeline_options import PdfPipelineOptions
PDF_PIPELINE_OPTIONS = PdfPipelineOptions()
PDF_PIPELINE_OPTIONS.do_code_enrichment = False
PDF_PIPELINE_OPTIONS.do_ocr = True
PDF_PIPELINE_OPTIONS.do_table_extraction = True

In [2]:
parser = LlamaParse(
    api_key=os.environ['LLAMA_CLOUD_API_KEY'],
    result_type="markdown",  # Critical for tables
    parsing_instruction="""Parse Material Test Certificates with these requirements:
    - Preserve all table structures (chemical analysis, mechanical properties)
    - Keep numerical values exact with all decimals
    - Maintain relationships between headers and values
    - Extract metadata: MTR number, heat number, order number, dates, grades
    - Keep units associated with values (MPa, %, HB, etc.)"""
)

In [3]:
file_extractor = {".pdf": parser}
documents = SimpleDirectoryReader(
    input_dir="./mtc_documents/",
    file_extractor=file_extractor
).load_data()


2025-11-19 07:52:54,092 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"


Started parsing the file under job_id 1a0c04ee-5c2e-4cb0-972a-9d588d79d8ff


2025-11-19 07:52:55,374 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/1a0c04ee-5c2e-4cb0-972a-9d588d79d8ff "HTTP/1.1 200 OK"
2025-11-19 07:52:57,885 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/1a0c04ee-5c2e-4cb0-972a-9d588d79d8ff "HTTP/1.1 200 OK"
2025-11-19 07:52:59,139 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/1a0c04ee-5c2e-4cb0-972a-9d588d79d8ff/result/markdown "HTTP/1.1 200 OK"


In [4]:
print(f"Loaded {len(documents)} MTC documents")

Loaded 2 MTC documents


In [5]:
# Use MarkdownElementNodeParser - it understands table boundaries
node_parser = MarkdownElementNodeParser(
    llm=OpenAI(model="gpt-4o",api_key=os.environ['OPEN_API_KEY']),
    num_workers=8
)


In [6]:
nodes = node_parser.get_nodes_from_documents(documents)
base_nodes, objects = node_parser.get_nodes_and_objects(nodes)

print(f"Created {len(base_nodes)} base nodes and {len(objects)} table objects")

0it [00:00, ?it/s]
0it [00:00, ?it/s]


Created 2 base nodes and 0 table objects


In [9]:
base_index = VectorStoreIndex(base_nodes,embed_model=OpenAIEmbedding(api_key=os.environ['OPEN_API_KEY']))  
base_retriever = base_index.as_retriever(similarity_top_k=5)



2025-11-19 07:57:39,270 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [10]:
recursive_retriever = RecursiveRetriever(
    "vector",
    retriever_dict={"vector": base_retriever},
    node_dict={node.node_id: node for node in objects},
    verbose=True
)


In [11]:
query_engine = RetrieverQueryEngine.from_args(
    recursive_retriever,
    llm=OpenAI(model="gpt-4o",api_key=os.environ['OPEN_API_KEY']),
)


In [12]:
response = query_engine.query(
    "What is the carbon content in heat number 795247?"
)
print(response)

Retrieving with query id None: What is the carbon content in heat number 795247?


2025-11-19 07:58:12,789 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Retrieving text node: ```json
{
"metadata": {
"MTR_number": "67/215674/9",
"heat_number": "716392",
"order_number": "562196",
"sales_order_number": "43215674/00452",
"date": "18.01.2024",
"grade": "ASTM A182 UNS S31803/F51"
},
"chemical_analysis": {
"headers": [
"ELEMENT",
"%C",
"%P",
"%Mn",
"%Si",
"%S",
"%Cr",
"%Ni",
"%Mo",
"%N"
],
"values": [
{
"ELEMENT": "C",
"value": "0.022"
},
{
"ELEMENT": "P",
"value": "0.009"
},
{
"ELEMENT": "Mn",
"value": "1.14"
},
{
"ELEMENT": "Si",
"value": "0.49"
},
{
"ELEMENT": "S",
"value": "0.15"
},
{
"ELEMENT": "Cr",
"value": "21.32"
},
{
"ELEMENT": "Ni",
"value": "5.27"
},
{
"ELEMENT": "Mo",
"value": "0.20"
},
{
"ELEMENT": "N",
"value": "0.08"
}
],
"specification": {
"min": {
"%C": "0.030",
"%P": "0.020",
"%Mn": "3.2",
"%Si": "2.00",
"%S": "0.030",
"%Cr": "23.00",
"%Ni": "6.50",
"%Mo": "3.50",
"%N": "0.20"
},
"max": {
"%C": "0.022",
"%P": "0.009",
"%Mn": "1.14",
"%Si": "0.49",
"%S": "0.15",
"%Cr": "21.32",
"%Ni": "5.27",
"%Mo": "0.20",
"%N": "0.08"
}
}


2025-11-19 07:58:14,944 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The carbon content in heat number 795247 is 0.022%.


In [17]:
response = query_engine.query(
    "Tell me heat no of the order no 634728 "
)
print(response)

Retrieving with query id None: Tell me heat no of the order no 634728 


2025-11-19 08:06:58,537 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Retrieving text node: ```json
{
"metadata": {
"MTR_number": "67/215674/9",
"heat_number": "716392",
"order_number": "562196",
"sales_order_number": "43215674/00452",
"date": "18.01.2024",
"grade": "ASTM A182 UNS S31803/F51"
},
"chemical_analysis": {
"headers": [
"ELEMENT",
"%C",
"%P",
"%Mn",
"%Si",
"%S",
"%Cr",
"%Ni",
"%Mo",
"%N"
],
"values": [
{
"ELEMENT": "C",
"value": "0.022"
},
{
"ELEMENT": "P",
"value": "0.009"
},
{
"ELEMENT": "Mn",
"value": "1.14"
},
{
"ELEMENT": "Si",
"value": "0.49"
},
{
"ELEMENT": "S",
"value": "0.15"
},
{
"ELEMENT": "Cr",
"value": "21.32"
},
{
"ELEMENT": "Ni",
"value": "5.27"
},
{
"ELEMENT": "Mo",
"value": "0.20"
},
{
"ELEMENT": "N",
"value": "0.08"
}
],
"specification": {
"min": {
"%C": "0.030",
"%P": "0.020",
"%Mn": "3.2",
"%Si": "2.00",
"%S": "0.030",
"%Cr": "23.00",
"%Ni": "6.50",
"%Mo": "3.50",
"%N": "0.20"
},
"max": {
"%C": "0.022",
"%P": "0.009",
"%Mn": "1.14",
"%Si": "0.49",
"%S": "0.15",
"%Cr": "21.32",
"%Ni": "5.27",
"%Mo": "0.20",
"%N": "0.08"
}
}


2025-11-19 08:06:59,453 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The heat number for order number 634728 is 795247.


In [18]:
response = query_engine.query(
    "Which MTCs have tensile strength above 620 MPa?"
)
print(response)


Retrieving with query id None: Which MTCs have tensile strength above 620 MPa?


2025-11-19 08:10:11,764 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Retrieving text node: ```json
{
"metadata": {
"MTR_number": "68/292645/3",
"heat_number": "795247",
"order_number": "634728",
"dates": {
"issue_date": "24.01.2024",
"expiry_date": "21.01.2019"
},
"grades": "UNS S31803/F51"
},
"chemical_analysis": {
"headers": ["ELEMENT", "%C", "%Mn", "%Si", "%P", "%S", "%Mo", "%Ni", "%Cr", "%N"],
"values": [
["MIN", 0.17, 0.20, 0.08, 0.030, 0.009, 0.025, 6.50, 3.50, 0.20],
["MAX", 0.030, 2.00, 1.00, 0.030, 0.020, 0.025, 6.50, 3.50, 0.20],
["Actual", 0.022, 1.14, 0.49, 0.017, 0.009, 0.025, 6.50, 3.50, 0.20]
]
},
"mechanical_properties": {
"headers": ["PROPERTY", "VALUE", "UNIT"],
"values": [
["Tensile Strength", 620, "MPa"],
["Yield Strength (Rp=0.2%)", 450, "MPa"],
["Elongation", 21.00, "%"],
["Reduction of Area", 25.00, "%"],
["Hardness", 239, "HBW"],
["Charpy V-Notch", 195, "Joules"]
]
},
"dimensional_specifications": {
"description": "Round Flanges",
"size": "6\"",
"pressure_class": "150#",
"inspection": "100% inspected"
},
"additional_information":

2025-11-19 08:10:15,068 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The MTC with the MTR number "67/215674/9" has a tensile strength of 620 MPa, which is not above 620 MPa. The MTC with the MTR number "68/292645/3" also has a tensile strength of 620 MPa, which is not above 620 MPa. Therefore, none of the MTCs have a tensile strength above 620 MPa.
